# Module 10: Production Platform Skeleton

**Companion notebook for**: `10-production-platform.html`

## Learning Objectives
- Structure a FastAPI application for LLM serving
- Implement authentication middleware
- Build a token-bucket rate limiter
- Create a request router for model selection
- Implement semantic caching with embedding similarity
- Track costs per request and per user
- Add structured logging and health check endpoints

## Prerequisites
- Python 3.10+
- `ANTHROPIC_API_KEY` environment variable set (optional -- notebook uses mock fallbacks)
- Basic understanding of HTTP APIs and middleware patterns

---
## Setup & Dependencies

In [ ]:
%pip install -q anthropic pydantic

In [ ]:
import os
import json
import time
import hashlib
import math
import uuid
import logging
from datetime import datetime, timezone
from dataclasses import dataclass, field
from typing import Optional, Any
from collections import defaultdict

API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
USE_LIVE_API = bool(API_KEY)

print(f"Live API available: {USE_LIVE_API}")
if not USE_LIVE_API:
    print("Running in mock/simulation mode -- all outputs are simulated.")

---
## Section 1: FastAPI App Structure

Define the core application structure with Pydantic models for
request/response validation. This is the skeleton that all other
components plug into.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal


# --- Request / Response Models ---

class ChatRequest(BaseModel):
    """Incoming chat completion request."""
    messages: list[dict] = Field(min_length=1, description="List of {role, content} messages")
    model: str = Field(default="auto", description="Model name or 'auto' for routing")
    max_tokens: int = Field(default=1024, ge=1, le=4096)
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)
    user_id: Optional[str] = None


class ChatResponse(BaseModel):
    """Chat completion response."""
    id: str = Field(description="Unique request ID")
    model: str = Field(description="Model that handled the request")
    content: str = Field(description="Generated response text")
    usage: dict = Field(description="Token usage statistics")
    cached: bool = Field(default=False, description="Whether response came from cache")
    latency_ms: float = Field(description="Total latency in milliseconds")


class HealthResponse(BaseModel):
    """Health check response."""
    status: Literal["healthy", "degraded", "unhealthy"]
    version: str
    uptime_seconds: float
    checks: dict[str, bool]


# --- Application Skeleton ---

APP_CONFIG = {
    "title": "LLM Gateway API",
    "version": "1.0.0",
    "description": "Production LLM serving platform with auth, rate limiting, caching, and cost tracking.",
    "routes": [
        {"method": "POST", "path": "/v1/chat/completions", "handler": "chat_completions"},
        {"method": "GET",  "path": "/health",              "handler": "health_check"},
        {"method": "GET",  "path": "/v1/models",           "handler": "list_models"},
        {"method": "GET",  "path": "/v1/usage",            "handler": "get_usage"},
    ],
    "middleware": ["auth", "rate_limit", "logging", "cost_tracking"],
}

print("FastAPI Application Structure")
print("=" * 50)
print(f"  Title: {APP_CONFIG['title']}")
print(f"  Version: {APP_CONFIG['version']}")
print(f"\n  Routes:")
for route in APP_CONFIG["routes"]:
    print(f"    {route['method']:<6} {route['path']:<30} -> {route['handler']}")
print(f"\n  Middleware stack:")
for i, mw in enumerate(APP_CONFIG["middleware"], 1):
    print(f"    {i}. {mw}")

# Show what the FastAPI code would look like
print(f"\n  FastAPI app definition:")
print("  " + "-" * 40)
print("  from fastapi import FastAPI, Depends")
print('  app = FastAPI(title="LLM Gateway API")')
print('  app.add_middleware(AuthMiddleware)')
print('  app.add_middleware(RateLimitMiddleware)')
print('  @app.post("/v1/chat/completions")')
print('  async def chat(req: ChatRequest): ...')

---
## Section 2: Authentication Middleware

API key-based authentication that validates requests before they reach
the handler. Supports multiple API keys with different permission levels.

In [ ]:
@dataclass
class APIKeyInfo:
    """Metadata associated with an API key."""
    key_id: str
    user_id: str
    tier: Literal["free", "standard", "premium"]
    rate_limit_rpm: int   # Requests per minute
    allowed_models: list[str]
    active: bool = True


class AuthMiddleware:
    """API key authentication middleware."""
    
    def __init__(self):
        # In production, these come from a database
        self._keys: dict[str, APIKeyInfo] = {
            "sk-test-free-001": APIKeyInfo(
                key_id="key_001", user_id="user_free", tier="free",
                rate_limit_rpm=10, allowed_models=["claude-haiku"],
            ),
            "sk-test-standard-002": APIKeyInfo(
                key_id="key_002", user_id="user_standard", tier="standard",
                rate_limit_rpm=60, allowed_models=["claude-haiku", "claude-sonnet"],
            ),
            "sk-test-premium-003": APIKeyInfo(
                key_id="key_003", user_id="user_premium", tier="premium",
                rate_limit_rpm=300, allowed_models=["claude-haiku", "claude-sonnet", "claude-opus"],
            ),
        }
    
    def authenticate(self, api_key: Optional[str]) -> tuple[bool, Optional[APIKeyInfo], str]:
        """
        Validate an API key.
        Returns: (success, key_info, error_message)
        """
        if not api_key:
            return False, None, "Missing API key. Set Authorization: Bearer <key>"
        
        key_info = self._keys.get(api_key)
        if key_info is None:
            return False, None, "Invalid API key"
        
        if not key_info.active:
            return False, None, "API key has been deactivated"
        
        return True, key_info, ""


# Test authentication
auth = AuthMiddleware()

test_keys = [
    "sk-test-free-001",
    "sk-test-premium-003",
    "sk-invalid-key",
    None,
]

print("Authentication Test")
print("=" * 60)
for key in test_keys:
    success, info, error = auth.authenticate(key)
    display_key = key[:20] + "..." if key and len(key) > 20 else str(key)
    if success:
        print(f"  [{display_key}] -> AUTHENTICATED")
        print(f"    User: {info.user_id}, Tier: {info.tier}, RPM: {info.rate_limit_rpm}")
        print(f"    Models: {', '.join(info.allowed_models)}")
    else:
        print(f"  [{display_key}] -> DENIED: {error}")

---
## Section 3: Rate Limiter

Token bucket rate limiter that controls how many requests a user can make
per time window. Each tier has different limits.

In [ ]:
@dataclass
class TokenBucket:
    """Token bucket for rate limiting."""
    capacity: float          # Max tokens (requests)
    tokens: float            # Current tokens
    refill_rate: float       # Tokens added per second
    last_refill: float       # Timestamp of last refill


class RateLimiter:
    """Per-user token bucket rate limiter."""
    
    def __init__(self):
        self._buckets: dict[str, TokenBucket] = {}
    
    def _get_bucket(self, user_id: str, rpm: int) -> TokenBucket:
        """Get or create a token bucket for a user."""
        if user_id not in self._buckets:
            self._buckets[user_id] = TokenBucket(
                capacity=float(rpm),
                tokens=float(rpm),
                refill_rate=rpm / 60.0,  # Tokens per second
                last_refill=time.time(),
            )
        return self._buckets[user_id]
    
    def _refill(self, bucket: TokenBucket):
        """Refill tokens based on elapsed time."""
        now = time.time()
        elapsed = now - bucket.last_refill
        bucket.tokens = min(bucket.capacity, bucket.tokens + elapsed * bucket.refill_rate)
        bucket.last_refill = now
    
    def check(self, user_id: str, rpm: int) -> tuple[bool, dict]:
        """
        Check if a request is allowed.
        Returns: (allowed, info_dict)
        """
        bucket = self._get_bucket(user_id, rpm)
        self._refill(bucket)
        
        info = {
            "remaining": int(bucket.tokens),
            "limit": int(bucket.capacity),
            "reset_seconds": round((bucket.capacity - bucket.tokens) / bucket.refill_rate, 1) if bucket.refill_rate > 0 else 0,
        }
        
        if bucket.tokens >= 1.0:
            bucket.tokens -= 1.0
            info["remaining"] = int(bucket.tokens)
            return True, info
        else:
            return False, info


# Test rate limiter
limiter = RateLimiter()

print("Rate Limiter Test")
print("=" * 50)

# Simulate a free-tier user hitting the limit (10 RPM)
user = "user_free"
rpm = 10
print(f"\nUser: {user} (limit: {rpm} RPM)")
print(f"{'Request':>8} {'Allowed':>8} {'Remaining':>10} {'Reset (s)':>10}")
print("-" * 40)

for i in range(1, 13):
    allowed, info = limiter.check(user, rpm)
    status = "YES" if allowed else "NO"
    print(f"{i:>8} {status:>8} {info['remaining']:>10} {info['reset_seconds']:>10}")

# Show a premium user with higher limits
print(f"\nUser: user_premium (limit: 300 RPM)")
for i in range(1, 4):
    allowed, info = limiter.check("user_premium", 300)
    print(f"  Request {i}: {'ALLOWED' if allowed else 'DENIED'} (remaining: {info['remaining']})")

---
## Section 4: Request Router

The router selects the best model for a request based on the task,
user tier, and cost constraints. When `model='auto'`, it routes
automatically.

In [ ]:
@dataclass
class ModelConfig:
    """Configuration for an available model."""
    name: str
    provider: str
    cost_per_1k_input: float
    cost_per_1k_output: float
    max_context: int
    speed_tier: Literal["fast", "medium", "slow"]
    quality_tier: Literal["standard", "high", "premium"]


AVAILABLE_MODELS = {
    "claude-haiku": ModelConfig(
        name="claude-haiku", provider="anthropic",
        cost_per_1k_input=0.00025, cost_per_1k_output=0.00125,
        max_context=200000, speed_tier="fast", quality_tier="standard",
    ),
    "claude-sonnet": ModelConfig(
        name="claude-sonnet", provider="anthropic",
        cost_per_1k_input=0.003, cost_per_1k_output=0.015,
        max_context=200000, speed_tier="medium", quality_tier="high",
    ),
    "claude-opus": ModelConfig(
        name="claude-opus", provider="anthropic",
        cost_per_1k_input=0.015, cost_per_1k_output=0.075,
        max_context=200000, speed_tier="slow", quality_tier="premium",
    ),
}


class RequestRouter:
    """Route requests to the appropriate model."""
    
    # Keywords that suggest complex tasks needing higher-quality models
    COMPLEX_KEYWORDS = ["analyze", "compare", "explain in detail", "write code",
                        "debug", "architecture", "design", "review"]
    
    def route(self, request: ChatRequest, key_info: APIKeyInfo) -> ModelConfig:
        """Select the best model for a request."""
        if request.model != "auto":
            # User explicitly requested a model
            if request.model in AVAILABLE_MODELS:
                model = AVAILABLE_MODELS[request.model]
                # Check if user tier allows this model
                if request.model in key_info.allowed_models:
                    return model
            # Fall through to auto-routing
        
        # Auto-routing based on content complexity
        user_msg = ""
        for msg in request.messages:
            if msg.get("role") == "user":
                user_msg = msg.get("content", "")
        
        is_complex = any(kw in user_msg.lower() for kw in self.COMPLEX_KEYWORDS)
        is_long = len(user_msg) > 500
        
        # Route based on complexity and user tier
        if key_info.tier == "premium" and (is_complex or is_long):
            return AVAILABLE_MODELS.get("claude-opus", AVAILABLE_MODELS["claude-sonnet"])
        elif is_complex or is_long:
            if "claude-sonnet" in key_info.allowed_models:
                return AVAILABLE_MODELS["claude-sonnet"]
        
        # Default: cheapest allowed model
        for model_name in ["claude-haiku", "claude-sonnet", "claude-opus"]:
            if model_name in key_info.allowed_models:
                return AVAILABLE_MODELS[model_name]
        
        return AVAILABLE_MODELS["claude-haiku"]


# Test routing
router = RequestRouter()

test_cases = [
    ("What is 2+2?", "free", "auto"),
    ("Analyze the performance implications of this architecture design.", "standard", "auto"),
    ("Hello", "premium", "auto"),
    ("Write code for a distributed cache system with proper error handling.", "premium", "auto"),
    ("Hi there", "standard", "claude-sonnet"),
]

print("Request Routing")
print("=" * 70)
print(f"{'User Msg (truncated)':<45} {'Tier':<10} {'Routed To':<15} {'Quality'}")
print("-" * 70)

for msg, tier, model in test_cases:
    # Create matching key info
    if tier == "free":
        ki = APIKeyInfo(key_id="k", user_id="u", tier="free", rate_limit_rpm=10, allowed_models=["claude-haiku"])
    elif tier == "standard":
        ki = APIKeyInfo(key_id="k", user_id="u", tier="standard", rate_limit_rpm=60, allowed_models=["claude-haiku", "claude-sonnet"])
    else:
        ki = APIKeyInfo(key_id="k", user_id="u", tier="premium", rate_limit_rpm=300, allowed_models=["claude-haiku", "claude-sonnet", "claude-opus"])
    
    req = ChatRequest(messages=[{"role": "user", "content": msg}], model=model)
    routed = router.route(req, ki)
    print(f"{msg[:43]:<45} {tier:<10} {routed.name:<15} {routed.quality_tier}")

---
## Section 5: Semantic Cache

A semantic cache stores responses keyed by the meaning of the query,
not just exact text matches. We use simple embedding similarity to
find cached responses for similar queries.

In [ ]:
class SemanticCache:
    """
    Cache LLM responses with semantic similarity matching.
    Uses a simplified bag-of-words embedding for demonstration.
    In production, use a proper embedding model (e.g., text-embedding-3-small).
    """
    
    def __init__(self, similarity_threshold: float = 0.85, max_entries: int = 1000):
        self.threshold = similarity_threshold
        self.max_entries = max_entries
        self._cache: list[dict] = []  # [{"embedding": [...], "query": str, "response": str, "model": str}]
    
    def _embed(self, text: str) -> list[float]:
        """
        Simple bag-of-words embedding (normalized term frequency).
        In production, use: openai.embeddings.create() or similar.
        """
        # Build vocabulary from all cached + new text
        words = text.lower().split()
        # Use word hashing for fixed-size vector
        dim = 128
        vec = [0.0] * dim
        for word in words:
            idx = hash(word) % dim
            vec[idx] += 1.0
        # Normalize
        magnitude = math.sqrt(sum(v*v for v in vec))
        if magnitude > 0:
            vec = [v / magnitude for v in vec]
        return vec
    
    def _cosine_similarity(self, a: list[float], b: list[float]) -> float:
        """Compute cosine similarity between two vectors."""
        dot = sum(x*y for x, y in zip(a, b))
        mag_a = math.sqrt(sum(x*x for x in a))
        mag_b = math.sqrt(sum(x*x for x in b))
        if mag_a == 0 or mag_b == 0:
            return 0.0
        return dot / (mag_a * mag_b)
    
    def get(self, query: str) -> Optional[dict]:
        """Look up a cached response by semantic similarity."""
        query_emb = self._embed(query)
        
        best_match = None
        best_sim = 0.0
        
        for entry in self._cache:
            sim = self._cosine_similarity(query_emb, entry["embedding"])
            if sim > best_sim:
                best_sim = sim
                best_match = entry
        
        if best_match and best_sim >= self.threshold:
            return {
                "response": best_match["response"],
                "model": best_match["model"],
                "similarity": round(best_sim, 4),
                "original_query": best_match["query"],
            }
        return None
    
    def put(self, query: str, response: str, model: str):
        """Store a response in the cache."""
        if len(self._cache) >= self.max_entries:
            self._cache.pop(0)  # Evict oldest
        
        self._cache.append({
            "embedding": self._embed(query),
            "query": query,
            "response": response,
            "model": model,
        })
    
    @property
    def size(self) -> int:
        return len(self._cache)


# Test semantic cache
cache = SemanticCache(similarity_threshold=0.80)

# Populate cache
cache.put("What is machine learning?", "Machine learning is a subset of AI that learns from data.", "claude-haiku")
cache.put("Explain transformers in NLP", "Transformers use self-attention to process sequences in parallel.", "claude-sonnet")
cache.put("How does Python garbage collection work?", "Python uses reference counting and a cyclic garbage collector.", "claude-haiku")

# Test lookups
test_queries = [
    "What is ML?",                          # Similar to cached query
    "Tell me about machine learning",        # Semantically similar
    "Explain the transformer architecture",  # Similar to cached
    "What is quantum computing?",            # No match
    "How does garbage collection work in Python?",  # Very similar
]

print("Semantic Cache Test")
print("=" * 70)
print(f"Cache size: {cache.size} entries\n")

for query in test_queries:
    result = cache.get(query)
    if result:
        print(f"  HIT  [{result['similarity']:.2f}] '{query}'")
        print(f"       Matched: '{result['original_query']}'")
        print(f"       Response: {result['response'][:60]}...")
    else:
        print(f"  MISS [----] '{query}'")
    print()

---
## Section 6: Cost Tracker

Track per-request and per-user costs based on token usage and model pricing.
Essential for billing, budgeting, and cost optimization.

In [ ]:
@dataclass
class CostEntry:
    """A single cost record."""
    request_id: str
    user_id: str
    model: str
    input_tokens: int
    output_tokens: int
    cost_usd: float
    timestamp: str
    cached: bool = False


class CostTracker:
    """Track and report API usage costs."""
    
    def __init__(self):
        self._entries: list[CostEntry] = []
    
    def record(self, request_id: str, user_id: str, model: str,
               input_tokens: int, output_tokens: int, cached: bool = False) -> CostEntry:
        """Record a request's cost."""
        model_config = AVAILABLE_MODELS.get(model)
        if model_config and not cached:
            cost = (
                (input_tokens / 1000) * model_config.cost_per_1k_input +
                (output_tokens / 1000) * model_config.cost_per_1k_output
            )
        else:
            cost = 0.0  # Cached responses are free
        
        entry = CostEntry(
            request_id=request_id,
            user_id=user_id,
            model=model,
            input_tokens=input_tokens,
            output_tokens=output_tokens,
            cost_usd=round(cost, 6),
            timestamp=datetime.now(timezone.utc).isoformat(),
            cached=cached,
        )
        self._entries.append(entry)
        return entry
    
    def get_user_summary(self, user_id: str) -> dict:
        """Get cost summary for a specific user."""
        user_entries = [e for e in self._entries if e.user_id == user_id]
        if not user_entries:
            return {"user_id": user_id, "total_cost": 0, "requests": 0}
        
        return {
            "user_id": user_id,
            "total_cost": round(sum(e.cost_usd for e in user_entries), 6),
            "total_requests": len(user_entries),
            "cached_requests": sum(1 for e in user_entries if e.cached),
            "total_input_tokens": sum(e.input_tokens for e in user_entries),
            "total_output_tokens": sum(e.output_tokens for e in user_entries),
            "by_model": {
                model: {
                    "requests": sum(1 for e in user_entries if e.model == model),
                    "cost": round(sum(e.cost_usd for e in user_entries if e.model == model), 6),
                }
                for model in set(e.model for e in user_entries)
            },
        }
    
    def get_global_summary(self) -> dict:
        """Get cost summary across all users."""
        return {
            "total_cost": round(sum(e.cost_usd for e in self._entries), 6),
            "total_requests": len(self._entries),
            "unique_users": len(set(e.user_id for e in self._entries)),
            "cache_hit_rate": (
                sum(1 for e in self._entries if e.cached) / len(self._entries)
                if self._entries else 0
            ),
        }


# Simulate some requests
import random
random.seed(42)

cost_tracker = CostTracker()

sim_requests = [
    ("user_free", "claude-haiku", 100, 200, False),
    ("user_free", "claude-haiku", 150, 300, False),
    ("user_free", "claude-haiku", 100, 200, True),   # Cached
    ("user_standard", "claude-sonnet", 500, 800, False),
    ("user_standard", "claude-haiku", 200, 400, False),
    ("user_standard", "claude-sonnet", 500, 800, True),  # Cached
    ("user_premium", "claude-opus", 1000, 2000, False),
    ("user_premium", "claude-sonnet", 800, 1500, False),
    ("user_premium", "claude-opus", 1000, 2000, True),  # Cached
    ("user_premium", "claude-haiku", 100, 200, False),
]

for user, model, inp, out, cached in sim_requests:
    cost_tracker.record(
        request_id=str(uuid.uuid4())[:8],
        user_id=user,
        model=model,
        input_tokens=inp,
        output_tokens=out,
        cached=cached,
    )

# Print reports
print("Cost Tracking Report")
print("=" * 60)

global_summary = cost_tracker.get_global_summary()
print(f"\nGlobal Summary:")
print(f"  Total cost:     ${global_summary['total_cost']:.4f}")
print(f"  Total requests: {global_summary['total_requests']}")
print(f"  Unique users:   {global_summary['unique_users']}")
print(f"  Cache hit rate: {global_summary['cache_hit_rate']:.0%}")

for user_id in ["user_free", "user_standard", "user_premium"]:
    summary = cost_tracker.get_user_summary(user_id)
    print(f"\n  {user_id}:")
    print(f"    Total cost: ${summary['total_cost']:.4f}")
    print(f"    Requests: {summary['total_requests']} ({summary['cached_requests']} cached)")
    print(f"    Tokens: {summary['total_input_tokens']} in / {summary['total_output_tokens']} out")
    for model, stats in summary["by_model"].items():
        print(f"      {model}: {stats['requests']} reqs, ${stats['cost']:.4f}")

---
## Section 7: Structured Logging & Health Check

Production systems need structured JSON logs for observability
and health check endpoints for monitoring and load balancers.

In [ ]:
class StructuredLogger:
    """JSON structured logger for production observability."""
    
    def __init__(self, service_name: str = "llm-gateway"):
        self.service_name = service_name
        self._logs: list[dict] = []
    
    def log(self, level: str, event: str, **kwargs):
        """Create a structured log entry."""
        entry = {
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "level": level,
            "service": self.service_name,
            "event": event,
            **kwargs,
        }
        self._logs.append(entry)
        return entry
    
    def info(self, event: str, **kwargs):
        return self.log("INFO", event, **kwargs)
    
    def warn(self, event: str, **kwargs):
        return self.log("WARN", event, **kwargs)
    
    def error(self, event: str, **kwargs):
        return self.log("ERROR", event, **kwargs)


class HealthChecker:
    """Health check endpoint logic."""
    
    def __init__(self, version: str = "1.0.0"):
        self.version = version
        self.start_time = time.time()
    
    def check(self) -> HealthResponse:
        """Run all health checks and return status."""
        checks = {
            "api_keys_loaded": True,     # Auth system is working
            "rate_limiter": True,        # Rate limiter is working
            "cache_available": True,     # Cache is accessible
            "model_reachable": USE_LIVE_API,  # Can reach LLM API
        }
        
        all_healthy = all(checks.values())
        critical_healthy = checks["api_keys_loaded"] and checks["rate_limiter"]
        
        if all_healthy:
            status = "healthy"
        elif critical_healthy:
            status = "degraded"
        else:
            status = "unhealthy"
        
        return HealthResponse(
            status=status,
            version=self.version,
            uptime_seconds=round(time.time() - self.start_time, 2),
            checks=checks,
        )


# Demo structured logging
logger = StructuredLogger()

# Simulate request lifecycle logging
request_id = str(uuid.uuid4())[:8]
logger.info("request_received", request_id=request_id, user_id="user_premium", model="auto")
logger.info("auth_success", request_id=request_id, tier="premium")
logger.info("rate_limit_check", request_id=request_id, remaining=298)
logger.info("model_routed", request_id=request_id, model="claude-sonnet", reason="complex_query")
logger.info("cache_miss", request_id=request_id)
logger.info("llm_response", request_id=request_id, latency_ms=1250, input_tokens=500, output_tokens=800)
logger.info("cache_stored", request_id=request_id)
logger.info("request_complete", request_id=request_id, total_latency_ms=1280, cost_usd=0.0135)

print("Structured Log Output")
print("=" * 70)
for entry in logger._logs:
    print(json.dumps(entry))

# Health check
print(f"\n\nHealth Check Response")
print("=" * 50)
health = HealthChecker(version="1.0.0")
status = health.check()
print(json.dumps(status.model_dump(), indent=2))

---
## Section 8: Full Request Pipeline Simulation

Tie all components together and simulate a complete request flowing
through the production platform.

In [ ]:
class LLMGateway:
    """
    Complete production LLM gateway.
    Ties together: auth, rate limiting, routing, caching, cost tracking, logging.
    """
    
    def __init__(self):
        self.auth = AuthMiddleware()
        self.limiter = RateLimiter()
        self.router = RequestRouter()
        self.cache = SemanticCache(similarity_threshold=0.85)
        self.cost_tracker = CostTracker()
        self.logger = StructuredLogger()
        self.health = HealthChecker()
    
    def handle_request(self, api_key: str, request: ChatRequest) -> dict:
        """Process a complete request through all middleware."""
        request_id = str(uuid.uuid4())[:8]
        start = time.time()
        
        self.logger.info("request_received", request_id=request_id)
        
        # 1. Authentication
        auth_ok, key_info, auth_err = self.auth.authenticate(api_key)
        if not auth_ok:
            self.logger.warn("auth_failed", request_id=request_id, error=auth_err)
            return {"error": auth_err, "status": 401}
        self.logger.info("auth_success", request_id=request_id, user=key_info.user_id)
        
        # 2. Rate limiting
        rate_ok, rate_info = self.limiter.check(key_info.user_id, key_info.rate_limit_rpm)
        if not rate_ok:
            self.logger.warn("rate_limited", request_id=request_id, user=key_info.user_id)
            return {"error": "Rate limit exceeded", "status": 429, "retry_after": rate_info["reset_seconds"]}
        
        # 3. Route to model
        model = self.router.route(request, key_info)
        self.logger.info("model_routed", request_id=request_id, model=model.name)
        
        # 4. Check cache
        user_msg = next((m["content"] for m in request.messages if m["role"] == "user"), "")
        cached = self.cache.get(user_msg)
        
        if cached:
            self.logger.info("cache_hit", request_id=request_id, similarity=cached["similarity"])
            latency = (time.time() - start) * 1000
            self.cost_tracker.record(request_id, key_info.user_id, model.name, 0, 0, cached=True)
            return {
                "status": 200,
                "response": ChatResponse(
                    id=request_id, model=cached["model"], content=cached["response"],
                    usage={"input_tokens": 0, "output_tokens": 0}, cached=True, latency_ms=round(latency, 1),
                ).model_dump(),
            }
        
        # 5. Call LLM (simulated)
        self.logger.info("cache_miss", request_id=request_id)
        input_tokens = len(user_msg.split()) * 2  # Rough estimate
        output_tokens = 150  # Simulated
        response_text = f"Simulated response from {model.name} for: {user_msg[:50]}"
        
        # 6. Cache the response
        self.cache.put(user_msg, response_text, model.name)
        
        # 7. Track cost
        self.cost_tracker.record(request_id, key_info.user_id, model.name, input_tokens, output_tokens)
        
        latency = (time.time() - start) * 1000
        self.logger.info("request_complete", request_id=request_id, latency_ms=round(latency, 1))
        
        return {
            "status": 200,
            "response": ChatResponse(
                id=request_id, model=model.name, content=response_text,
                usage={"input_tokens": input_tokens, "output_tokens": output_tokens},
                cached=False, latency_ms=round(latency, 1),
            ).model_dump(),
        }


# Run simulation
gateway = LLMGateway()

sim_requests = [
    ("sk-test-premium-003", "Analyze the transformer architecture in detail."),
    ("sk-test-free-001",    "What is 2+2?"),
    ("sk-invalid-key",      "This should fail."),
    ("sk-test-standard-002","Analyze the transformer architecture in detail."),  # Cache hit candidate
    ("sk-test-premium-003", "Write code for a REST API."),
]

print("Full Pipeline Simulation")
print("=" * 70)

for api_key, user_msg in sim_requests:
    req = ChatRequest(messages=[{"role": "user", "content": user_msg}])
    result = gateway.handle_request(api_key, req)
    
    print(f"\nKey: {api_key[:20]}... | Msg: {user_msg[:40]}")
    print(f"  Status: {result['status']}")
    if result["status"] == 200:
        resp = result["response"]
        print(f"  Model: {resp['model']} | Cached: {resp['cached']} | Latency: {resp['latency_ms']:.1f}ms")
    else:
        print(f"  Error: {result.get('error', 'Unknown')}")

# Final summary
print(f"\n\nGateway Summary")
print("=" * 40)
summary = gateway.cost_tracker.get_global_summary()
print(f"  Total requests: {summary['total_requests']}")
print(f"  Total cost: ${summary['total_cost']:.4f}")
print(f"  Cache hit rate: {summary['cache_hit_rate']:.0%}")
print(f"  Cache entries: {gateway.cache.size}")

---
## Summary

In this notebook we built a complete production LLM platform skeleton:

1. **FastAPI Structure** -- Request/response models with Pydantic validation, route definitions, and middleware stack.

2. **Authentication** -- API key validation with tiered access (free/standard/premium), model restrictions, and rate limit configuration.

3. **Rate Limiting** -- Token bucket algorithm that controls per-user request rates with automatic refill.

4. **Request Routing** -- Intelligent model selection based on query complexity, user tier, and cost. Supports explicit model selection and auto-routing.

5. **Semantic Cache** -- Embedding-based cache that matches semantically similar queries, reducing cost and latency for repeated questions.

6. **Cost Tracking** -- Per-request, per-user, and per-model cost tracking based on token usage and model pricing. Cache hits are free.

7. **Structured Logging** -- JSON-formatted log entries for observability, with request tracing via request IDs.

8. **Health Check** -- Endpoint for monitoring and load balancer integration, reporting system status and component health.

**Key insight**: A production LLM platform is mostly infrastructure -- auth, rate limiting, caching, logging, and cost tracking. The actual LLM call is a small part of the system. Building these components correctly is what makes an LLM application production-ready.

**Architecture pattern**: Request -> Auth -> Rate Limit -> Route -> Cache Check -> LLM Call -> Cache Store -> Cost Track -> Log -> Response.